# Aprendizado de Máquina — Aula prática 06

## Pré-processamento e Pipelines

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Nos notebooks das aulas anteriores, por muitas vezes encontramos construção assim:

```python
Pipeline([("escala", StandardScaler()), ("modelo", Ridge())])
```

e, ao lado dela, um comentário dizendo que a Aula 06 explicaria o que era aquilo.
Este é o notebook em que a explicação acontece.

A pergunta que organiza tudo o que vem a seguir é simples:

> **em que momento eu devo calcular a média e o desvio para padronizar as
> covariáveis — antes ou depois de separar treino e teste?**

Você talvez já saiba a resposta de cor ("depois"). O que este notebook acrescenta é
o *porquê* e, principalmente, o *quanto*: vamos medir o tamanho do estrago de fazer
errado, e o número costuma surpreender.

O caminho tem quatro paradas. Primeiro descobrimos **para quem** a escala das
covariáveis importa, porque não é para todo mundo. Depois entendemos **por que**
padronizar traz um risco embutido. Em seguida medimos **quanto** esse risco custa.
E no fim montamos o objeto que elimina o risco por construção.

### Objetivos

Ao terminar este notebook, você deve conseguir responder sem hesitar:

- dado um método qualquer, ele muda de resposta se eu trocar a unidade de uma
  coluna? E qual é a pergunta que decide isso?
- por que a padronização, que parece uma operação inofensiva, pode inflar as minhas
  métricas?
- o que exatamente um `Pipeline` garante — e o que ele **não** garante?
- como faço o `GridSearchCV` enxergar um parâmetro escondido dentro de uma etapa do
  `Pipeline`?

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd

Duas importações são as protagonistas da aula:

- o **`StandardScaler`** é quem padroniza. Para cada coluna ele guarda a média e o
  desvio-padrão, e depois transforma cada valor em $(x - \mu)/\sigma$;
- o **`Pipeline`** é quem encadeia. Ele junta transformações e o estimador final
  num objeto só, que se comporta como se fosse um estimador comum.

O resto já apareceu em aulas anteriores. O `make_regression` fabrica um problema de
regressão com as características que a gente pedir, e só entra em cena na Seção 5.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. Quem é sensível à escala, e quem não é

"Padronize sempre" é um conselho que circula bastante. O problema dele não é estar
errado — é não dizer **para quem** vale, nem **por quê**. Quem segue conselho sem
entender acaba padronizando onde não fazia falta e esquecendo onde fazia.

Dá para descobrir por conta própria, e a ideia do experimento é esta: pegamos um
problema qualquer e mudamos a **unidade** de uma única coluna, multiplicando-a por
1000, como quem anota em milímetros o que estava anotado em metros.

Essa troca não acrescenta nem tira informação alguma. A coluna continua dizendo
exatamente a mesma coisa sobre o mundo; mudou só o número em que ela está escrita.
Em princípio, portanto, nenhum método deveria se importar.

**Antes de rodar a célula, arrisque um palpite.** Dos quatro métodos abaixo — MQO,
Ridge, árvore de regressão e KNN —, quais vão dar uma resposta diferente depois da
troca de unidade?

A célula ajusta cada um deles duas vezes, uma em cada escala, e mede o risco das
duas versões contra a função de regressão verdadeira. Se um método for indiferente
à unidade, os dois números saem iguais.

In [ ]:
rng = np.random.default_rng(0)
n, d = 400, 6


def alvo(M):
    return M[:, 0] + 0.5 * M[:, 1] - M[:, 2] + 0.3 * M[:, 3] * M[:, 4]


X = rng.normal(size=(n, d))
y = alvo(X) + rng.normal(0, 0.5, size=n)
X_te = rng.normal(size=(3000, d))
r_te = alvo(X_te)

# a mesma informacao, com a coluna 0 em outra unidade
escala = np.ones(d)
escala[0] = 1000.0
Xe, Xe_te = X * escala, X_te * escala

metodos = {
    "MQO": skl.LinearRegression(),
    "Ridge (alpha=100)": skl.Ridge(alpha=100.0),
    "arvore": DecisionTreeRegressor(max_depth=6, random_state=0),
    "KNN (k=10)": KNeighborsRegressor(n_neighbors=10),
}

linhas = []
for nome, m in metodos.items():
    a = np.mean((m.fit(X, y).predict(X_te) - r_te) ** 2)
    b = np.mean((m.fit(Xe, y).predict(Xe_te) - r_te) ** 2)
    linhas.append({"metodo": nome, "escala original": a, "coluna 0 x1000": b,
                   "mudou?": "sim" if abs(b - a) > 1e-6 * max(a, 1) else "nao"})

# o coeficiente da coluna 0 na Ridge, nas duas escalas (reescalado para comparar)
c1 = skl.Ridge(alpha=100.0).fit(X, y).coef_[0]
c2 = skl.Ridge(alpha=100.0).fit(Xe, y).coef_[0] * 1000
print(f"Ridge: coeficiente da coluna 0 = {c1:.4f} na escala original, "
      f"{c2:.4f} depois de multiplicar a coluna por 1000")
pd.DataFrame(linhas).set_index("metodo").round(4)

Dois métodos ficaram parados e dois se mexeram. Vamos por partes, porque cada
resposta tem um motivo próprio — e é o motivo que interessa levar daqui, não a
lista.

**O MQO não muda: $0{,}0914$ nas duas escalas.** A razão é curta. Se a coluna $j$
passa a valer $1000\,x_j$, basta o coeficiente passar a valer $\beta_j/1000$ para o
produto $\beta_j x_j$ continuar o mesmo — e, com ele, toda a predição. Como os
mínimos quadrados só olham para os resíduos, e os resíduos não mudaram, é
exatamente esse coeficiente que o método encontra. Diz-se que o MQO é
**equivariante** por reescala. Na prática: padronizar antes de um MQO puro não
altera predição nenhuma.

**A árvore também não muda: $0{,}6032$ nas duas.** Aqui o motivo é outro. A árvore
nunca soma colunas; ela só faz perguntas do tipo "$x_j \le t$?". Multiplicar a
coluna por 1000 multiplica o corte $t$ por 1000 junto, e a pergunta separa
exatamente as mesmas observações. O que importa é a ordem dos valores dentro da
coluna, e a ordem não mudou.

**A Ridge muda**, e o coeficiente impresso acima mostra por quê. A Ridge não
minimiza só os resíduos: ela minimiza os resíduos **mais** a penalidade
$\lambda\sum_j \beta_j^2$. O problema está nessa soma, que junta coeficientes de
colunas diferentes — somar um coeficiente medido "por metro" com outro medido "por
real" não significa grande coisa.

Acompanhe pelos números. Na escala original, o coeficiente verdadeiro da coluna 0
vale $1{,}0$, e a penalidade o encolhe até $0{,}79$. Depois de multiplicar a coluna
por 1000, o coeficiente correspondente fica mil vezes menor, da ordem de
$0{,}001$ — e um número tão pequeno quase não pesa em $\sum_j \beta_j^2$. Resultado:
ele escapa quase intacto, chegando a $0{,}99$ quando trazido de volta à escala
comparável. **A mesma coluna, a mesma informação, e uma quantidade de regularização
completamente diferente.**

**O KNN muda muito**, de $0{,}35$ para $1{,}46$. A distância euclidiana entre duas
observações é $\sum_j (x_{ij} - x_{kj})^2$: também uma soma sobre colunas. Se uma
coluna fica mil vezes maior, as diferenças nela ficam mil vezes maiores e os
quadrados, um milhão de vezes. Ela passa a decidir sozinha quem é vizinho de quem, e
as outras cinco colunas somem da conta.

Junte os quatro casos e a regra aparece sozinha. Todos os que mudaram têm em comum
uma **soma sobre colunas diferentes** em algum lugar: na penalidade (Ridge, Lasso),
na distância (KNN, SVM, $k$-médias) ou na projeção (PCA). Os que não mudaram nunca
somam colunas.

A regra, então, não é "padronize sempre". É: **padronize quando o método somar
coisas vindas de colunas diferentes.** Nos outros casos padronizar não faz mal — só
não faz nada.

Uma regra boa serve para **prever** o comportamento de um método que você ainda não
mediu, só olhando o mecanismo dele. Vamos testar a nossa em dois casos:

- o **Lasso** penaliza $\lambda\sum_j |\beta_j|$, que é uma soma sobre colunas
  diferentes. A regra prevê que ele **muda**;
- a **floresta aleatória** é um conjunto de árvores, e árvores só comparam valores
  dentro de cada coluna. A regra prevê que ela **não muda**.

A célula abaixo mede os dois e imprime também o coeficiente da coluna 0 no Lasso,
para vermos o mecanismo funcionando.

In [ ]:
for nome, m in [("Lasso (alpha=0,1)", skl.Lasso(alpha=0.1)),
                ("floresta (B=200)", RandomForestRegressor(n_estimators=200,
                                                           random_state=0, n_jobs=-1))]:
    risco_a = np.mean((m.fit(X, y).predict(X_te) - r_te) ** 2)
    coef_a = getattr(m, "coef_", None)
    risco_b = np.mean((m.fit(Xe, y).predict(Xe_te) - r_te) ** 2)
    coef_b = getattr(m, "coef_", None)

    print(f"{nome:20s} original {risco_a:.4f}   reescalado {risco_b:.4f}"
          f"   ({100 * (risco_b - risco_a) / risco_a:+.1f}%)")
    if coef_a is not None:
        print(f"{'':20s} coef da coluna 0: {coef_a[0]:.4f} -> {coef_b[0]:.6f}"
              f"   nao-nulos: {(coef_a != 0).sum()} -> {(coef_b != 0).sum()}")

As duas previsões se confirmam, e a da floresta se confirma de forma bem
convincente: $0{,}2421$ nas duas escalas, os mesmos quatro dígitos — não um valor
parecido, o mesmo valor. A indiferença que a árvore isolada mostrou na seção
anterior atravessa intacta a agregação de 200 árvores.

O Lasso muda, e o coeficiente conta a história. Ele sai de $0{,}8846$ para
$0{,}000983$: exatamente mil vezes menor, como tem de ser para o produto
$\beta_0 x_0$ continuar o mesmo. Só que agora esse coeficiente minúsculo entra em
$\sum_j|\beta_j|$ valendo quase nada. A coluna 0 ficou praticamente **isenta** da
penalidade, enquanto as outras cinco continuam pagando integralmente.

Aí vem o detalhe que mais confunde quem vê isso pela primeira vez: o risco
**melhorou** 9,2% depois da reescala. Isso quer dizer que não padronizar foi uma boa
ideia?

Não. Pense no que aconteceu. A coluna 0 é justamente a de maior efeito verdadeiro
nesta simulação, e a reescala a isentou da penalidade. Isentar de penalidade a
coluna que mais importa calhou de ajudar — desta vez. Se a mesma reescala tivesse
caído sobre uma coluna irrelevante, o efeito seria o oposto: ela ficaria livre para
receber um coeficiente grande sem pagar nada por isso, e o ajuste pioraria.

Nos dois casos o ponto é o mesmo, e é o argumento central a favor da padronização:
**quem escolheu quanto regularizar cada coluna não foi você, foi a unidade em que
os dados chegaram.**

---
## 3. A desvantagem do `StandardScaler`: $\mu$ e $\sigma$ são aprendidos

A Seção 2 mostrou por que padronizar é necessário. Esta mostra o preço que se paga
por isso — e é o assunto central da aula.

Comece pela definição. Padronizar a coluna $j$ é substituir cada valor por

$$\widetilde{x}_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}.$$

A pergunta que interessa é: de onde vêm $\mu_j$ e $\sigma_j$? Eles não são
constantes conhecidas do problema. São a média e o desvio **da amostra** — isto é,
são *estimados a partir dos dados*, exatamente como um coeficiente de regressão é.

E é essa palavra, *estimados*, que cria o problema. Se você ajustar o
`StandardScaler` no conjunto inteiro **antes** de separar treino e teste, os
$\widehat\mu_j$ e $\widehat\sigma_j$ que o treino usa terão sido calculados com a
ajuda das observações do teste. As métricas deixam de medir desempenho em dados
verdadeiramente novos e ficam otimistas. É o fenômeno que se chama **vazamento de
dados** (*data leakage*).

O mesmo raciocínio vale dobra a dobra dentro de uma validação cruzada: em cada
dobra, o scaler deveria ser reajustado usando só o treino *daquela* dobra.

Até aqui é o aviso que qualquer livro dá. O que quase nenhum livro faz é dizer
**quanto** isso custa — e é o que a célula abaixo mede, repetindo a comparação 40
vezes para que o resultado não dependa de um sorteio de sorte.

O cenário foi escolhido para ser o mais favorável possível ao alarme: um problema
com sinal de verdade e oito colunas cujas escalas diferem por quatro ordens de
grandeza, indo de $0{,}01$ a $200$. Se padronizar fora da dobra for perigoso em
algum lugar, é aqui.

**Antes de rodar: quanto você aposta que o $R^2$ vai inflar?**

In [ ]:
rng_e = np.random.default_rng(6)
unidades = np.array([1, 50, 0.01, 5, 1, 200, 0.1, 2])
beta = np.r_[1.5, 0.02, 80, 0.3, -1.0, 0.005, 10, 0.4]
dobras = skm.KFold(5, shuffle=True, random_state=0)

fora, dentro = [], []
for _ in range(40):
    Xp = rng_e.normal(size=(60, 8)) * unidades
    yp = Xp @ beta + rng_e.normal(0, 1, 60)

    # ERRADO: o scaler aprende media e desvio no conjunto TODO
    esc = StandardScaler().fit(Xp)
    fora.append(skm.cross_val_score(skl.Ridge(alpha=1.0), esc.transform(Xp), yp,
                                    cv=dobras, scoring="r2").mean())

    # CERTO: o scaler e uma etapa do pipeline, reajustada dentro de cada dobra
    dentro.append(skm.cross_val_score(
        Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge(alpha=1.0))]),
        Xp, yp, cv=dobras, scoring="r2").mean())

print(f"escala FORA da dobra  : R^2 = {np.mean(fora):.4f}")
print(f"escala DENTRO do tubo : R^2 = {np.mean(dentro):.4f}")
print(f"diferenca             : {abs(np.mean(fora) - np.mean(dentro)):.2e}")

$0{,}8342$ contra $0{,}8341$.

Os dois coincidem até a terceira casa decimal. A diferença entre a conduta errada e
a conduta certa é da ordem de $10^{-4}$ — e isso no cenário que montamos justamente
para exagerar o problema.

**Por que tão pouco?** Pense no que o `StandardScaler` de fato expõe quando enxerga
o conjunto todo: **dois números por coluna**, uma média e um desvio, cada um
calculado sobre dezenas de observações. Trocar a média de 48 observações pela média
de 60 desloca o valor por algo da ordem de $1/\sqrt{n}$ — e desloca **todas** as
observações do mesmo jeito, sem olhar para o $y$ de nenhuma delas. Não existe aí um
canal pelo qual o modelo possa aprender algo específico sobre as observações de
validação.

Isso não significa que vazamento seja um problema inventado. Significa que **este**
vazamento é pequeno. O caro mora em outro lugar: em escolher *variáveis* ou
*hiperparâmetros* olhando a resposta. As notas desta aula medem esse caso, e a
distância entre os dois é de quatro ordens de grandeza.

Qual é a conduta, então? A mesma de sempre, por um motivo que não tem nada a ver com
o tamanho do estrago: **corrija porque corrigir custa zero.** Basta pôr o
`StandardScaler` dentro de um `Pipeline` — que é o que a próxima seção faz. Não há
por que debater se um erro barato é aceitável quando o acerto sai de graça.

---
## 4. O `Pipeline`, por dentro

Na seção anterior o `Pipeline` apareceu resolvendo o problema, mas sem explicação.
Vamos abrir o objeto, porque o que ele garante é bem preciso — e convém saber o que
é, para saber também o que **não** é.

Um `Pipeline` é uma lista de pares `(nome, transformador)`, terminada por um
estimador. O que ele faz é disciplinar quem chama `fit` em quê:

- quando você escreve `pipe.fit(X_tr, y_tr)`, cada transformador da lista faz
  `fit_transform` **nos dados de treino**, em sequência, e o estimador final é
  ajustado no resultado;
- quando você escreve `pipe.predict(X_te)`, cada transformador faz apenas
  `transform`, usando os parâmetros que já havia aprendido, e o estimador prediz.

A consequência é que o conjunto de teste nunca participa de nenhum `fit`. E isso não
depende da sua disciplina nem da sua memória: é uma propriedade da estrutura.

Afirmação dessas se verifica. A célula abaixo ajusta um `Pipeline` e depois pergunta
ao `StandardScaler` de dentro dele qual média ele guardou, comparando com a média do
treino e com a do conjunto todo. Só uma das duas pode bater — qual?

In [ ]:
X_tr, X_va, y_tr, y_va = skm.train_test_split(Xe, y, test_size=0.3, random_state=0)

pipe = Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge(alpha=1.0))])
pipe.fit(X_tr, y_tr)

media_aprendida = pipe.named_steps["escala"].mean_
print("media que o scaler aprendeu (3 primeiras colunas):",
      np.round(media_aprendida[:3], 4))
print("media do TREINO                                  :",
      np.round(X_tr.mean(axis=0)[:3], 4))
print("media do conjunto TODO                           :",
      np.round(Xe.mean(axis=0)[:3], 4))
print(f"\nbate com o treino? {np.allclose(media_aprendida, X_tr.mean(axis=0))}")

A média guardada pelo scaler bate com a do treino, e não com a do conjunto todo. As
duas são bem diferentes na primeira coluna, $-33{,}94$ contra $-89{,}02$, e isso não
é coincidência: é a coluna que multiplicamos por 1000 na Seção 2, então qualquer
diferença entre médias aparece amplificada nela.

Há um segundo ponto, que é o que torna o `Pipeline` prático de verdade: ele **é** um
estimador. Tem `fit`, tem `predict`, e por isso pode ser passado a qualquer coisa
que espere um estimador — `cross_val_score`, `GridSearchCV`, um `BaggingRegressor`.
A próxima seção vive disso.

E o que ele **não** garante? O `Pipeline` controla *quando* cada etapa é ajustada.
Ele não tem opinião nenhuma sobre *como* as suas observações foram repartidas entre
as dobras. Se duas linhas do seu banco se referirem à mesma pessoa, ele não vai
perceber — e esse é um problema que as notas da aula tratam à parte.

---
## 5. `Pipeline` $+$ `GridSearchCV`

Chegamos ao motivo pelo qual valeu a pena montar tudo isso.

Escolher um hiperparâmetro por validação cruzada significa ajustar o modelo muitas
vezes: uma por dobra e por candidato. Se houver pré-processamento envolvido, ele
precisa ser refeito em cada um desses ajustes — e fazer isso à mão, corretamente,
dezenas de vezes seguidas, é justamente onde os erros nascem.

A solução é entregar o `Pipeline` inteiro ao `GridSearchCV`, em vez de entregar só o
estimador. A busca passa então a tratar o pré-processamento como parte do modelo,
que é o que ele é.

Falta um detalhe de sintaxe. Como dizer ao `GridSearchCV` que queremos variar o
`alpha` que está *dentro* da etapa chamada `lasso`? Com dois sublinhados:

```python
{"lasso__alpha": [0.001, 0.01, 0.1, 1, 5, 10, 50, 100]}
```

Leia da esquerda para a direita: `lasso` é o nome que demos à etapa, `__` desce um
nível, `alpha` é o parâmetro lá dentro. A mesma regra encadeia quantos níveis forem
necessários.

O problema abaixo é feito sob medida para Lasso e ElasticNet: mil observações, cem
covariáveis, e só dez delas realmente informativas.

In [ ]:
X_reg, y_reg = make_regression(n_samples=1000, n_features=100, n_informative=10,
                              n_targets=1, noise=1.0, random_state=0)
X_tr2, X_te2, y_tr2, y_te2 = skm.train_test_split(X_reg, y_reg, test_size=0.3,
                                                  random_state=0)

tubo_lasso = Pipeline([("escala", StandardScaler()), ("lasso", skl.Lasso())])
tubo_enet = Pipeline([("escala", StandardScaler()), ("enet", skl.ElasticNet())])

grade_lasso = {"lasso__alpha": [0.001, 0.01, 0.1, 1, 5, 10, 50, 100]}
grade_enet = {"enet__alpha": [0.001, 0.01, 0.1, 1, 5, 10, 50, 100],
              "enet__l1_ratio": [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1]}

for nome, tubo, grade in [("Lasso", tubo_lasso, grade_lasso),
                          ("ElasticNet", tubo_enet, grade_enet)]:
    busca = skm.GridSearchCV(tubo, grade, cv=5,
                             scoring="neg_mean_squared_error").fit(X_tr2, y_tr2)
    print(f"{nome:11s} {len(busca.cv_results_['mean_test_score']):3d} combinacoes"
          f"   {busca.best_params_}")
    print(f"{'':11s} EQM de CV {-busca.best_score_:.4f}"
          f"   EQM no teste {mean_squared_error(y_te2, busca.predict(X_te2)):.4f}")

Os dois métodos chegaram ao mesmo lugar, e a razão é instrutiva.

O `ElasticNet` tem dois hiperparâmetros: o `alpha`, que mede a força da penalidade,
e o `l1_ratio`, que decide quanto dela é $\ell_1$ (Lasso) e quanto é $\ell_2$
(Ridge). A busca varreu $8 \times 7 = 56$ combinações, e a vencedora tem
`l1_ratio = 1` — ou seja, penalidade $\ell_1$ pura, que é exatamente o Lasso. Com
cem covariáveis das quais só dez importam, a busca redescobriu sozinha que o
caminho certo era zerar as outras noventa.

Agora compare os dois erros. O de validação cruzada foi $1{,}1293$; o do conjunto de
teste, $1{,}2110$ — pior. Isso era esperado, e o motivo vale para qualquer busca de
hiperparâmetro.

O `best_score_` é o **melhor** de 56 estimativas, e cada uma delas é ruidosa. O
mínimo de 56 sorteios ruidosos tende a ficar abaixo do valor verdadeiro, mesmo que
cada sorteio, isoladamente, seja não enviesado. Selecionar é otimizar, e quem
otimizou não pode reportar. Por isso a última linha usa `X_te2`, que o
`GridSearchCV` nunca viu: **a validação cruzada escolhe, o conjunto de teste mede.**

Para fechar, uma conta que dá a medida do trabalho que o `Pipeline` fez por você. A
padronização foi refeita $41$ vezes na busca do Lasso e $281$ na do ElasticNet — os
$8\times5$ e $56\times5$ ajustes das dobras, mais o reajuste final de cada busca no
conjunto de treino inteiro. Todas com a média e o desvio da dobra certa, e nenhuma
delas escrita por você.

---
## Resumo

| Pergunta | Onde respondemos | Resposta curta |
|---|---|---|
| Meu método muda se eu trocar a unidade de uma coluna? | §2 | muda se ele somar coisas de colunas diferentes: penalidade, distância, projeção |
| Por que padronizar tem risco? | §3 | porque $\mu$ e $\sigma$ são *estimados*, e estimá-los no conjunto todo usa o teste |
| Quanto custa esse risco? | §3 | pouco: $0{,}8342$ contra $0{,}8341$. Mas corrigir custa zero |
| O que o `Pipeline` garante? | §4 | que o teste nunca entra num `fit` — por estrutura, não por disciplina |
| E o que ele **não** garante? | §4 | nada sobre *como* as dobras foram formadas |
| Como alcanço um parâmetro de dentro dele? | §5 | `nomeDaEtapa__parametro`, com dois sublinhados |

Se sobrar uma única frase desta aula, que seja esta: **toda etapa que aprende algo
dos dados é parte do modelo**, e o `Pipeline` é como se diz isso em código.

**Leitura recomendada.** [ISLP] os laboratórios dos Capítulos 5 e 6, onde o
`Pipeline` aparece pela primeira vez; [AME] §2.1.2. E a documentação do
`scikit-learn`, *Pipelines and composite estimators*.

**Para praticar.** `Lista teorica 06.pdf` (teórica, com gabarito) e
`Lista prática 06.ipynb` (prática, para completar as lacunas), nesta mesma pasta.
As notas da aula vão além deste laboratório: elas medem também o vazamento por
seleção de variáveis e o vazamento por observações agrupadas, e ordenam os três por
gravidade.

**A seguir.** Fecha o Bloco I. A Aula 07 abre a classificação: a resposta deixa de
ser um número e passa a ser uma classe, o risco deixa de ser erro quadrático, e
quase tudo o que construímos até aqui precisa ser reescrito — com a vantagem de que
agora sabemos exatamente o que estamos reescrevendo.